# QA Checkpoint: Phases 1-3

This notebook performs quality assurance checks on the completed Phases 1-3 of the project.

## Phases Overview
- **Phase 1**: Business Understanding
- **Phase 2**: Data Understanding  
- **Phase 3**: Data Preprocessing

## Checks Performed
1. File existence verification
2. Module import testing
3. Cleaned data loading and validation
4. Data quality checks (missing values, duplicates)
5. Cross-dataset consistency checks
6. Summary report

## 1. Imports and Setup

In [ ]:
import pandas as pd
import sys
import os
from pathlib import Path

# Add project root to path for imports
sys.path.append('..')

# Expected file paths
PROJECT_ROOT = Path('..')
phase_files = {
    'Phase 1': [
        'Documents/Business_understanding.md'
    ],
    'Phase 2': [
        'Documents/Data_understanding.md',
        'notebooks/01_Business_Understanding.ipynb',
        'notebooks/02_Data_Understanding.ipynb'
    ],
    'Phase 3': [
        'src/preprocess.py',
        'notebooks/03_Preprocessing.ipynb',
        'data/processed/product_info_clean.csv',
        'data/processed/weekly_sales_clean.csv',
        'data/processed/external_features_clean.csv'
    ]
}

# Expected shapes
expected_shapes = {
    'product_info_clean.csv': (25, 10),
    'weekly_sales_clean.csv': (2600, 7),
    'external_features_clean.csv': (104, 6)
}

print("Setup complete")

Setup complete


## 2. File Existence Verification

In [2]:
file_checks = []

for phase, files in phase_files.items():
    for file_path in files:
        exists = (PROJECT_ROOT / file_path).exists()
        file_checks.append({
            'Phase': phase,
            'File': file_path,
            'Exists': 'PASS' if exists else 'FAIL'
        })

file_df = pd.DataFrame(file_checks)
print("File existence check results:")
file_df

File existence check results:


,Phase,File,Exists
0,Phase 1,Documents/Business_understanding.md,FAIL
1,Phase 2,Documents/Data_understanding.md,FAIL
2,Phase 2,notebooks/01_Business_Understanding.ipynb,FAIL
3,Phase 2,notebooks/02_Data_Understanding.ipynb,FAIL
4,Phase 3,src/preprocess.py,FAIL
5,Phase 3,notebooks/03_Preprocessing.ipynb,FAIL
6,Phase 3,data/processed/product_info_clean.csv,FAIL
7,Phase 3,data/processed/weekly_sales_clean.csv,FAIL
8,Phase 3,data/processed/external_features_clean.csv,FAIL


## 3. Module Import Testing

In [ ]:
import_checks = []

# Test src module imports
modules_to_test = [
    ('src.io', ['load_product_information', 'load_weekly_sales', 'load_external_features']),
    ('src.preprocess', ['preprocess_product_info', 'preprocess_weekly_sales', 'preprocess_external_features']),
    ('src.config', ['DATA_PROCESSED'])
]

for module_name, expected_items in modules_to_test:
    try:
        module = __import__(module_name, fromlist=expected_items)
        for item in expected_items:
            if hasattr(module, item):
                import_checks.append({
                    'Module': module_name,
                    'Item': item,
                    'Status': 'PASS'
                })
            else:
                import_checks.append({
                    'Module': module_name,
                    'Item': item,
                    'Status': 'FAIL'
                })
    except ImportError as e:
        for item in expected_items:
            import_checks.append({
                'Module': module_name,
                'Item': item,
                'Status': f'FAIL: {str(e)}'
            })

import_df = pd.DataFrame(import_checks)
print("Import check results:")
import_df

## 4. Load Cleaned Data

In [ ]:
# Load cleaned datasets
try:
    df_product = pd.read_csv(PROJECT_ROOT / 'data/processed/product_info_clean.csv')
    df_sales = pd.read_csv(PROJECT_ROOT / 'data/processed/weekly_sales_clean.csv')
    df_external = pd.read_csv(PROJECT_ROOT / 'data/processed/external_features_clean.csv')
    load_status = "PASS"
    print("All cleaned datasets loaded successfully")
except Exception as e:
    load_status = f"FAIL: {str(e)}"
    print(f"Error loading datasets: {e}")
    df_product = df_sales = df_external = None

## 5. Shape Validation

In [ ]:
shape_checks = []

if df_product is not None:
    actual_shape = df_product.shape
    expected = expected_shapes['product_info_clean.csv']
    shape_checks.append({
        'Dataset': 'product_info_clean.csv',
        'Expected Shape': expected,
        'Actual Shape': actual_shape,
        'Status': 'PASS' if actual_shape == expected else 'FAIL'
    })

if df_sales is not None:
    actual_shape = df_sales.shape
    expected = expected_shapes['weekly_sales_clean.csv']
    shape_checks.append({
        'Dataset': 'weekly_sales_clean.csv',
        'Expected Shape': expected,
        'Actual Shape': actual_shape,
        'Status': 'PASS' if actual_shape == expected else 'FAIL'
    })

if df_external is not None:
    actual_shape = df_external.shape
    expected = expected_shapes['external_features_clean.csv']
    shape_checks.append({
        'Dataset': 'external_features_clean.csv',
        'Expected Shape': expected,
        'Actual Shape': actual_shape,
        'Status': 'PASS' if actual_shape == expected else 'FAIL'
    })

shape_df = pd.DataFrame(shape_checks)
print("Shape validation results:")
shape_df

## 6. Missing Values Check

In [ ]:
missing_checks = []

datasets = [
    ('Product Info', df_product),
    ('Weekly Sales', df_sales),
    ('External Features', df_external)
]

for name, df in datasets:
    if df is not None:
        total_missing = df.isnull().sum().sum()
        missing_checks.append({
            'Dataset': name,
            'Total Missing Values': total_missing,
            'Status': 'PASS' if total_missing == 0 else 'FAIL'
        })
    else:
        missing_checks.append({
            'Dataset': name,
            'Total Missing Values': 'N/A',
            'Status': 'FAIL'
        })

missing_df = pd.DataFrame(missing_checks)
print("Missing values check results:")
missing_df

## 7. Duplicates Check

In [ ]:
duplicate_checks = []

for name, df in datasets:
    if df is not None:
        # Check for complete duplicates
        dup_count = df.duplicated().sum()
        duplicate_checks.append({
            'Dataset': name,
            'Duplicate Rows': dup_count,
            'Status': 'PASS' if dup_count == 0 else 'FAIL'
        })
    else:
        duplicate_checks.append({
            'Dataset': name,
            'Duplicate Rows': 'N/A',
            'Status': 'FAIL'
        })

duplicate_df = pd.DataFrame(duplicate_checks)
print("Duplicates check results:")
duplicate_df

## 8. Product_ID Consistency Check

In [ ]:
consistency_checks = []

if df_product is not None and df_sales is not None:
    product_ids_product = set(df_product['Product_ID'].dropna().unique())
    product_ids_sales = set(df_sales['Product_ID'].dropna().unique())
    
    missing_in_sales = product_ids_product - product_ids_sales
    extra_in_sales = product_ids_sales - product_ids_product
    
    consistency_checks.append({
        'Check': 'Product_ID in Product Info vs Weekly Sales',
        'Missing in Sales': len(missing_in_sales),
        'Extra in Sales': len(extra_in_sales),
        'Status': 'PASS' if len(missing_in_sales) == 0 and len(extra_in_sales) == 0 else 'FAIL'
    })
else:
    consistency_checks.append({
        'Check': 'Product_ID in Product Info vs Weekly Sales',
        'Missing in Sales': 'N/A',
        'Extra in Sales': 'N/A',
        'Status': 'FAIL'
    })

consistency_df = pd.DataFrame(consistency_checks)
print("Product_ID consistency check results:")
consistency_df

## 9. Week_End_Date Compatibility Check

In [ ]:
date_checks = []

if df_sales is not None and df_external is not None:
    # Convert to datetime if not already
    df_sales['Week_End_Date'] = pd.to_datetime(df_sales['Week_End_Date'], errors='coerce')
    df_external['Week_End_Date'] = pd.to_datetime(df_external['Week_End_Date'], errors='coerce')
    
    sales_dates = set(df_sales['Week_End_Date'].dropna().unique())
    external_dates = set(df_external['Week_End_Date'].dropna().unique())
    
    missing_in_external = sales_dates - external_dates
    extra_in_external = external_dates - sales_dates
    
    date_checks.append({
        'Check': 'Week_End_Date in Sales vs External',
        'Missing in External': len(missing_in_external),
        'Extra in External': len(extra_in_external),
        'Status': 'PASS' if len(missing_in_external) == 0 else 'WARN'  # Allow extra dates in external
    })
else:
    date_checks.append({
        'Check': 'Week_End_Date in Sales vs External',
        'Missing in External': 'N/A',
        'Extra in External': 'N/A',
        'Status': 'FAIL'
    })

date_df = pd.DataFrame(date_checks)
print("Week_End_Date compatibility check results:")
date_df

## 10. Summary Report

In [ ]:
# Combine all checks
all_checks = []

# File checks
for _, row in file_df.iterrows():
    all_checks.append({
        'Category': 'File Existence',
        'Check': f"{row['Phase']} - {row['File']}",
        'Status': row['Exists']
    })

# Import checks
for _, row in import_df.iterrows():
    all_checks.append({
        'Category': 'Import',
        'Check': f"{row['Module']}.{row['Item']}",
        'Status': row['Status']
    })

# Data loading
all_checks.append({
    'Category': 'Data Loading',
    'Check': 'Load cleaned CSVs',
    'Status': load_status
})

# Shape checks
for _, row in shape_df.iterrows():
    all_checks.append({
        'Category': 'Shape Validation',
        'Check': row['Dataset'],
        'Status': row['Status']
    })

# Missing values
for _, row in missing_df.iterrows():
    all_checks.append({
        'Category': 'Missing Values',
        'Check': row['Dataset'],
        'Status': row['Status']
    })

# Duplicates
for _, row in duplicate_df.iterrows():
    all_checks.append({
        'Category': 'Duplicates',
        'Check': row['Dataset'],
        'Status': row['Status']
    })

# Consistency
for _, row in consistency_df.iterrows():
    all_checks.append({
        'Category': 'Consistency',
        'Check': row['Check'],
        'Status': row['Status']
    })

# Date compatibility
for _, row in date_df.iterrows():
    all_checks.append({
        'Category': 'Date Compatibility',
        'Check': row['Check'],
        'Status': row['Status']
    })

summary_df = pd.DataFrame(all_checks)

# Calculate pass rate
total_checks = len(summary_df)
passed_checks = (summary_df['Status'] == 'PASS').sum()
pass_rate = passed_checks / total_checks * 100

print(f"QA Checkpoint Summary: {passed_checks}/{total_checks} checks passed ({pass_rate:.1f}%)")
print("\nDetailed Results:")
summary_df

## Conclusions

### Overall Status: [PASS/FAIL based on results]

### Key Findings:
- File existence: [Summary]
- Import functionality: [Summary]
- Data quality: [Summary]
- Cross-dataset consistency: [Summary]

### Recommendations for Next Phase:
- [Any issues found or confirmation all good]

### Ready for Phase 4: [Yes/No]